# Exercise 3.2: Efficiencies with TEfficiency revised

From *Programming in High Energy Particle Physics*, Chapter 3

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch03/ex03_2_solution.ipynb)

Exercise 3.2 Efficiencies with TEfficiency revised Extend the tree of Exercise 3.1 with the true Z rapidity \(y_Z\) and, for every lepton, a toy isolation variable (for instance a relative isolation drawn from an exponential distribution whose mean decreases with \(p_T\)). Using RDataFrame, (a) compute the efficiency of the selection \(|\eta| &lt; 2.4\) and isolation \(&lt; 0.15\) as a function of lepton \(p_T\), and (b) compute the acceptance, the fraction of Z decays in which both leptons have \(p_T &gt; 20\)&nbsp;GeV and \(|\eta| &lt; 2.4\), as a function of \(y_Z\). Plot both curves with TEfficiency .

<details><summary>Hint</summary>

Book two histograms with identical binning from the same RDataFrame node, one before and one after the selection ( Filter ), then build TEfficiency(*h_pass, *h_total) after dereferencing the results ( *h_pass gives the TH1D ). For per-lepton quantities, histogram the RVec columns directly or use Define with masks such as electron_pt[abs(electron_eta) &lt; 2.4] .

</details>

**Colab setup:** ROOT installation takes about 3–5 minutes and restarts the runtime once. After that restart, run all cells from the top again. The setup cells are safe to rerun.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    try:
        import ROOT
    except ImportError:
        get_ipython().run_line_magic("pip", "install -q condacolab")
        import condacolab
        condacolab.install()  # one-time runtime restart

In [ ]:
if IN_COLAB:
    try:
        import ROOT
    except ImportError:
        get_ipython().system("mamba install -q -y -c conda-forge root")
        import ROOT
else:
    import ROOT
ROOT.gROOT.SetBatch(True)

### Generate the tree

In [ ]:
import math
from array import array
from pathlib import Path
def create_tree(n_events=8000):
    rng=ROOT.TRandom3(42)
    f=ROOT.TFile("events.root","RECREATE")
    tree=ROOT.TTree("events","Toy dilepton events")
    floats={name:ROOT.std.vector('float')() for name in
        ('electron_pt','electron_eta','electron_phi','electron_iso',
         'muon_pt','muon_eta','muon_phi','muon_iso')}
    ints={name:ROOT.std.vector('int')() for name in ('electron_charge','muon_charge')}
    has_Z=array('i',[0]);z_pass=array('i',[0]);y_Z=array('f',[0.])
    for name,v in {**floats,**ints}.items():tree.Branch(name,v)
    tree.Branch('has_Z',has_Z,'has_Z/I');tree.Branch('z_pass',z_pass,'z_pass/I')
    tree.Branch('y_Z',y_Z,'y_Z/F')
    for _ in range(n_events):
        for v in [*floats.values(),*ints.values()]:v.clear()
        has_Z[0]=int(rng.Rndm()<0.35);z_pass[0]=0
        y_Z[0]=rng.Gaus(0,1.3) if has_Z[0] else 0.0
        for flavor in ('electron','muon'):
            for _ in range(rng.Integer(3)):
                pt=rng.Exp(25)+5;eta=rng.Gaus(0,1.8)
                floats[flavor+'_pt'].push_back(pt)
                floats[flavor+'_eta'].push_back(eta)
                floats[flavor+'_phi'].push_back(rng.Uniform(-math.pi,math.pi))
                floats[flavor+'_iso'].push_back(rng.Exp(0.25))
                ints[flavor+'_charge'].push_back(1 if rng.Rndm()<0.5 else -1)
        if has_Z[0]:
            flavor='electron' if rng.Rndm()<0.5 else 'muon'
            mass=max(70.,rng.Gaus(91.1876,2.8))
            costh=rng.Uniform(-0.98,0.98);eta_star=math.atanh(costh)
            pt_true=0.5*mass*math.sqrt(1-costh*costh)
            z_pass[0]=int(pt_true>20 and abs(y_Z[0]+eta_star)<2.4
                          and abs(y_Z[0]-eta_star)<2.4)
            phi=rng.Uniform(-math.pi,math.pi)
            for charge,eta,azimuth in ((1,y_Z[0]+eta_star,phi),
                                       (-1,y_Z[0]-eta_star,phi+math.pi)):
                pt=pt_true*rng.Gaus(1,0.02)
                floats[flavor+'_pt'].push_back(pt)
                floats[flavor+'_eta'].push_back(eta)
                floats[flavor+'_phi'].push_back(azimuth)
                floats[flavor+'_iso'].push_back(rng.Exp(0.12/(1+pt/50)))
                ints[flavor+'_charge'].push_back(charge)
        tree.Fill()
    tree.Write();f.Close()
    print(f"Created {n_events} toy events in events.root")
N_EVENTS=8000  # set to 50_000 for the full exercise
create_tree(N_EVENTS)

### Build TEfficiency curves

In [ ]:
df=ROOT.RDataFrame("events","events.root")
df=df.Define("e_pt_pass","electron_pt[(abs(electron_eta)<2.4) && (electron_iso<0.15)]")
df=df.Define("mu_pt_pass","muon_pt[(abs(muon_eta)<2.4) && (muon_iso<0.15)]")
h_e_tot=df.Histo1D(("e_tot",";Lepton p_{T} [GeV];Efficiency",20,0,100),"electron_pt").GetValue().Clone("e_tot_copy")
h_mu_tot=df.Histo1D(("mu_tot","",20,0,100),"muon_pt").GetValue()
h_e_tot.Add(h_mu_tot)
h_e_pass=df.Histo1D(("e_pass","",20,0,100),"e_pt_pass").GetValue().Clone("e_pass_copy")
h_mu_pass=df.Histo1D(("mu_pass","",20,0,100),"mu_pt_pass").GetValue()
h_e_pass.Add(h_mu_pass)
eff_pt=ROOT.TEfficiency(h_e_pass,h_e_tot)
signal=df.Filter("has_Z == 1")
h_y_tot=signal.Histo1D(("y_tot",";True Z rapidity;Acceptance",16,-4,4),"y_Z").GetValue()
h_y_pass=signal.Filter("z_pass == 1").Histo1D(("y_pass","",16,-4,4),"y_Z").GetValue()
eff_y=ROOT.TEfficiency(h_y_pass,h_y_tot)
c=ROOT.TCanvas("c_eff","Efficiency and acceptance",1000,450)
c.Divide(2,1);c.cd(1);eff_pt.Draw("AP");c.cd(2);eff_y.Draw("AP")
c.SaveAs("efficiencies.png")
print("Total signal events:",int(h_y_tot.GetEntries()))

In [ ]:
assert ROOT.TEfficiency.CheckConsistency(h_e_pass,h_e_tot)
assert ROOT.TEfficiency.CheckConsistency(h_y_pass,h_y_tot)

### Interpretation

Isolation improves with $p_T$ in this toy model, while acceptance falls at large $|y_Z|$. The qualitative book answer is reproduced; the precise curve is model dependent.

**Book answer:** (a) The efficiency rises with \(p_T\) as the toy isolation improves and flattens at the fraction of leptons inside \(|\eta| &lt; 2.4\). (b) The acceptance is largest for central Z bosons and drops towards zero for \(|y_Z| \gtrsim 2.5\), where at least one lepton almost always leaves the \(|\eta| &lt; 2.4\) region.